## Semantic Query Caching Simulation — DBpedia (`tab:caching`)

Implements the protocol in `notebooks/caching/README.md` exactly as committed
to in the paper (§Downstream Application):

- **Stream:** held-out SQuARE **test split** anchors, seeded shuffle.
- **Cache:** FIFO over the **K most recent** executed queries (embedding +
  result set). Hit when max cosine ≥ θ → return the cached result, do not
  insert. Miss → "execute" (oracle lookup) and insert.
- **Grid:** K ∈ {50, 100, 200} × θ ∈ {0.7, 0.8, 0.9}.
- **Models:** SparqlCE, TF-IDF, E5-base-v2, Qwen3-Embedding-0.6B.
- **Metrics:** hit rate; result quality = mean answer-set Jaccard between the
  returned (cached) result and the true result, over hits.

Requires the SparqlCE checkpoint `sparqlce/models/final_model_mrr_0.4487.pt`
(not in git — see `notebooks/visualization/FIGURE3_RUNBOOK.md`). Everything
else is in `datasets/`. Outputs `results/caching/caching_dbpedia.json` and
ready-to-paste LaTeX rows for `tab:caching`.


In [1]:
import gc
import os
import sys
import json
import random
from pathlib import Path
from collections import deque

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

# ---- Project root (works from notebooks/caching/ or anywhere in the repo) ----
_PROJECT_ROOT = Path.cwd()
while not (_PROJECT_ROOT / "datasets").exists():
    assert _PROJECT_ROOT != _PROJECT_ROOT.parent, "could not locate repo root"
    _PROJECT_ROOT = _PROJECT_ROOT.parent

if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))


# ---- Config ----
TRIPLETS_PATH = (
    _PROJECT_ROOT
    / "datasets"
    / "dbpedia-triplets-dataset.json"
)

SPLITS_PATH = (
    _PROJECT_ROOT
    / "datasets"
    / "splits_dbpedia.json"
)

RESULTS_PATH = (
    _PROJECT_ROOT
    / "datasets"
    / "result_sets_dbpedia.json"
)

SPARQLCE_CKPT = (
    _PROJECT_ROOT
    / "sparqlce"
    / "models"
    / "final_model_mrr_0.4487.pt"
)

OUT_DIR = (
    _PROJECT_ROOT
    / "results"
    / "caching"
)


SEED = 42

K_GRID = [
    50,
    100,
    200,
]

THETA_GRID = [
    0.7,
    0.8,
    0.9,
]

# (K, theta) row reported in the main table
HEADLINE = (100, 0.8)


DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    f"project root: {_PROJECT_ROOT}\n"
    f"device: {DEVICE}"
)

project root: /upb/users/g/ganeshk/profiles/unix/cs/SPARQL-Embeddings
device: cuda


/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/sparql/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# def normalize_query(q: str) -> str:
#     """Normalize formatting for exact oracle lookup only."""
#     return " ".join(q.split())


# with open(TRIPLETS_PATH, encoding="utf-8") as f:
#     triplets = json.load(f)

# with open(SPLITS_PATH, encoding="utf-8") as f:
#     splits = json.load(f)

# with open(RESULTS_PATH, encoding="utf-8") as f:
#     raw_result_sets = json.load(f)


# # Keep only non-empty answer sets and normalize query keys.
# result_sets = {
#     normalize_query(q): frozenset(rows)
#     for q, rows in raw_result_sets.items()
#     if rows
# }


# test_anchors = [
#     triplets[i]["anchor_query"]
#     for i in splits["test_idx"]
# ]

# train_anchors = [
#     triplets[i]["anchor_query"]
#     for i in splits["train_idx"]
# ]


# # Keep the ORIGINAL query strings in the stream for embedding.
# # Use normalized forms only to find oracle results.
# stream = [
#     q
#     for q in test_anchors
#     if normalize_query(q) in result_sets
# ]

# dropped = len(test_anchors) - len(stream)

# random.seed(SEED)
# random.shuffle(stream)

# print(
#     f"test anchors: {len(test_anchors)} | "
#     f"in stream: {len(stream)} | "
#     f"dropped (missing/empty oracle result): {dropped}"
# )


# def jaccard(a: frozenset, b: frozenset) -> float:
#     union = len(a | b)
#     return len(a & b) / union if union else 0.0


In [3]:
def normalize_query(q: str) -> str:
    """
    Normalize query formatting for exact oracle lookup only.

    The original query string is still used for embedding.
    """
    return " ".join(q.split())


def freeze_json(value):
    """
    Convert a JSON-compatible value into a recursively hashable form.

    This allows result rows represented as lists or dictionaries to be
    stored safely in a frozenset for answer-set Jaccard computation.
    """
    if isinstance(value, dict):
        return tuple(
            sorted(
                (
                    key,
                    freeze_json(item),
                )
                for key, item in value.items()
            )
        )

    if isinstance(value, list):
        return tuple(
            freeze_json(item)
            for item in value
        )

    return value


with open(
    TRIPLETS_PATH,
    encoding="utf-8",
) as f:
    triplets = json.load(f)


with open(
    SPLITS_PATH,
    encoding="utf-8",
) as f:
    splits = json.load(f)


with open(
    RESULTS_PATH,
    encoding="utf-8",
) as f:
    raw_result_sets = json.load(f)


# ---------------------------------------------------------------------------
# Build oracle lookup.
#
# Query keys are normalized only for exact oracle lookup.
# Result rows are converted into hashable canonical Python structures.
#
# A normalization collision with different result sets is treated as an error
# rather than silently overwriting one oracle entry with another.
# ---------------------------------------------------------------------------
result_sets = {}

for raw_query, rows in raw_result_sets.items():

    if not rows:
        continue

    query_key = normalize_query(raw_query)

    answer_set = frozenset(
        freeze_json(row)
        for row in rows
    )

    if (
        query_key in result_sets
        and result_sets[query_key] != answer_set
    ):
        raise ValueError(
            "Conflicting oracle result sets after query normalization:\n"
            f"{query_key}"
        )

    result_sets[query_key] = answer_set


# ---------------------------------------------------------------------------
# Build train/test anchor collections from the committed split.
# ---------------------------------------------------------------------------
test_anchors = [
    triplets[i]["anchor_query"]
    for i in splits["test_idx"]
]

train_anchors = [
    triplets[i]["anchor_query"]
    for i in splits["train_idx"]
]


# ---------------------------------------------------------------------------
# Keep ORIGINAL query strings in the stream for embedding.
#
# Normalized query strings are used only for oracle lookup.
# ---------------------------------------------------------------------------
stream = [
    q
    for q in test_anchors
    if normalize_query(q) in result_sets
]

dropped = (
    len(test_anchors)
    - len(stream)
)


# ---------------------------------------------------------------------------
# Shuffle the occurrence-level stream deterministically.
#
# Important:
# This deliberately does NOT deduplicate repeated anchors. If the same query
# occurs multiple times in the held-out workload, those occurrences remain
# separate stream events.
# ---------------------------------------------------------------------------
random.seed(SEED)
random.shuffle(stream)


# ---------------------------------------------------------------------------
# Diagnostics only. These values do not change the experiment.
# ---------------------------------------------------------------------------
stream_unique_queries = len(
    {
        normalize_query(q)
        for q in stream
    }
)

duplicate_stream_events = (
    len(stream)
    - stream_unique_queries
)


print(
    f"test anchors: {len(test_anchors)} | "
    f"in stream: {len(stream)} | "
    f"dropped (missing/empty oracle result): {dropped}"
)

print(
    f"unique normalized queries in stream: "
    f"{stream_unique_queries} | "
    f"repeated stream events: "
    f"{duplicate_stream_events}"
)


def jaccard(
    a: frozenset,
    b: frozenset,
) -> float:
    """
    Answer-set Jaccard similarity.
    """
    union = len(a | b)

    return (
        len(a & b) / union
        if union
        else 0.0
    )

test anchors: 472 | in stream: 396 | dropped (missing/empty oracle result): 76
unique normalized queries in stream: 396 | repeated stream events: 0


In [4]:
# ============================================================================
# 2. Embed the stream once per model
#
# All final stream embeddings are L2-normalised so that the dot product used
# in the cache simulation is cosine similarity.
# ============================================================================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sentence_transformers import SentenceTransformer

stream_embeddings = {}


# ---------------------------------------------------------------------------
# SparqlCE — trained checkpoint
# ---------------------------------------------------------------------------
from sparqlce.embedder import SPARQLEmbedder

sparqlce = SPARQLEmbedder(
    str(SPARQLCE_CKPT)
)

embs = []

for i in tqdm(
    range(0, len(stream), 32),
    desc="SparqlCE",
):
    embs.append(
        sparqlce.encode(
            stream[i : i + 32]
        )
    )

stream_embeddings["SparqlCE"] = normalize(
    np.vstack(embs),
    norm="l2",
    axis=1,
)

del sparqlce
del embs

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ---------------------------------------------------------------------------
# TF-IDF
#
# Fit on TRAIN anchors only.
# sklearn's TfidfVectorizer returns L2-normalised rows by default; norm="l2"
# is specified explicitly here so that the experimental assumption is visible.
# ---------------------------------------------------------------------------
tfidf = TfidfVectorizer(
    norm="l2"
).fit(
    train_anchors
)

stream_embeddings["TF-IDF"] = (
    tfidf
    .transform(stream)
    .toarray()
)


# ---------------------------------------------------------------------------
# E5-base-v2
#
# Zero-shot model usage with the "query: " prefix.
# ---------------------------------------------------------------------------
e5 = SentenceTransformer(
    "intfloat/e5-base-v2",
    device=DEVICE,
)

stream_embeddings["E5-base-v2"] = e5.encode(
    [
        f"query: {q}"
        for q in stream
    ],
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

del e5

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ---------------------------------------------------------------------------
# Qwen3-Embedding-0.6B
#
# Zero-shot, instruction-free usage. This preserves the behavior of the
# original experimental implementation.
# ---------------------------------------------------------------------------
qwen = SentenceTransformer(
    "Qwen/Qwen3-Embedding-0.6B",
    device=DEVICE,
)

stream_embeddings[
    "Qwen3-Embedding-0.6B"
] = qwen.encode(
    stream,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
    batch_size=8,
)

del qwen

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ---------------------------------------------------------------------------
# Sanity-check embedding matrix shapes.
# ---------------------------------------------------------------------------
for name, E in stream_embeddings.items():
    print(
        f"{name:>24}: {E.shape}"
    )

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 42067.76it/s]


Loading model from /upb/users/g/ganeshk/profiles/unix/cs/SPARQL-Embeddings/sparqlce/models/final_model_mrr_0.4487.pt...


Batches: 100%|██████████| 50/50 [00:01<00:00, 30.25it/s]


                SparqlCE: (396, 128)
                  TF-IDF: (396, 3670)
              E5-base-v2: (396, 768)
    Qwen3-Embedding-0.6B: (396, 1024)


In [5]:
# ============================================================================
# 3. Cache simulation
#
# Protocol:
#   - FIFO over the K most recent EXECUTED queries.
#   - Hit when max cosine similarity >= theta.
#   - On a hit: return the cached result and do not modify the cache.
#   - On a miss: use the oracle result and insert the query/result pair.
# ============================================================================

def simulate(
    stream,
    E,
    result_sets,
    K,
    theta,
):
    """
    Run one pass over the query stream.

    E[i] is the L2-normalised embedding of stream[i].
    """

    if len(stream) == 0:
        raise ValueError(
            "Cannot simulate an empty query stream."
        )

    if len(E) != len(stream):
        raise ValueError(
            "Embedding matrix length does not match stream length: "
            f"{len(E)} != {len(stream)}"
        )

    if K <= 0:
        raise ValueError(
            f"K must be positive, got {K}."
        )


    # Each entry is:
    #   (query_embedding, oracle_result_set)
    #
    # maxlen=K gives FIFO eviction automatically.
    cache = deque(
        maxlen=K
    )

    hits = 0
    qualities = []


    for i, q in enumerate(stream):

        R_true = result_sets[
            normalize_query(q)
        ]


        # -------------------------------------------------------------------
        # Cache lookup
        # -------------------------------------------------------------------
        if cache:

            cached_embeddings = np.stack(
                [
                    entry[0]
                    for entry in cache
                ]
            )

            sims = (
                cached_embeddings
                @ E[i]
            )

            best = int(
                np.argmax(sims)
            )


            # ---------------------------------------------------------------
            # Hit:
            #
            # Serve the nearest cached result.
            # Do not execute.
            # Do not insert.
            # Do not refresh cache position.
            # ---------------------------------------------------------------
            if float(sims[best]) >= theta:

                R_cached = cache[best][1]

                hits += 1

                qualities.append(
                    jaccard(
                        R_cached,
                        R_true,
                    )
                )

                continue


        # -------------------------------------------------------------------
        # Miss:
        #
        # "Execute" through the offline oracle and insert into FIFO cache.
        # -------------------------------------------------------------------
        cache.append(
            (
                E[i],
                R_true,
            )
        )


    n_queries = len(stream)

    return {
        "hit_rate": hits / n_queries,
        "result_quality": (
            float(np.mean(qualities))
            if qualities
            else None
        ),
        "n_hits": hits,
        "n_misses": n_queries - hits,
        "n_queries": n_queries,
    }


all_results = {}

for name, E in stream_embeddings.items():

    all_results[name] = {}

    for K in K_GRID:
        for theta in THETA_GRID:

            r = simulate(
                stream=stream,
                E=E,
                result_sets=result_sets,
                K=K,
                theta=theta,
            )

            all_results[name][
                f"K={K},theta={theta}"
            ] = r

    print(
        f"done: {name}"
    )

done: SparqlCE
done: TF-IDF
done: E5-base-v2
done: Qwen3-Embedding-0.6B


In [6]:
# ============================================================================
# 4. Save the full grid + display
# ============================================================================
OUT_DIR.mkdir(parents=True, exist_ok=True)
payload = {
    "kg": "dbpedia",
    "seed": SEED,
    "stream_length": len(stream),
    "dropped": dropped,
    "grid": {"K": K_GRID, "theta": THETA_GRID},
    "results": all_results,
}
out_json = OUT_DIR / "caching_dbpedia.json"
with open(out_json, "w") as f:
    json.dump(payload, f, indent=2)
print(f"saved {out_json}\n")

rows = []
for name, grid in all_results.items():
    for cfg, r in grid.items():
        K, theta = cfg.replace("K=", "").split(",theta=")
        rows.append({"model": name, "K": int(K), "theta": float(theta),
                     "hit_rate": r["hit_rate"], "result_quality": r["result_quality"]})
df = pd.DataFrame(rows)

print("Hit rate:")
print(df.pivot_table(index="model", columns=["K", "theta"], values="hit_rate").round(3))
print("\nResult quality (mean Jaccard over hits):")
print(df.pivot_table(index="model", columns=["K", "theta"], values="result_quality").round(3))

saved /upb/users/g/ganeshk/profiles/unix/cs/SPARQL-Embeddings/results/caching/caching_dbpedia.json

Hit rate:
K                       50                   100                  200         \
theta                   0.7    0.8    0.9    0.7    0.8    0.9    0.7    0.8   
model                                                                          
E5-base-v2            0.997  0.997  0.970  0.997  0.997  0.970  0.997  0.997   
Qwen3-Embedding-0.6B  0.674  0.210  0.008  0.730  0.303  0.013  0.730  0.346   
SparqlCE              0.821  0.624  0.278  0.836  0.710  0.351  0.836  0.720   
TF-IDF                0.795  0.753  0.644  0.826  0.788  0.662  0.826  0.788   

K                            
theta                   0.9  
model                        
E5-base-v2            0.970  
Qwen3-Embedding-0.6B  0.018  
SparqlCE              0.442  
TF-IDF                0.682  

Result quality (mean Jaccard over hits):
K                       50                   100                  200        

In [7]:
# ============================================================================
# 5. LaTeX rows for tab:caching
# ============================================================================

K_h, theta_h = HEADLINE


order = [
    "SparqlCE",
    "TF-IDF",
    "E5-base-v2",
    "Qwen3-Embedding-0.6B",
]


latex_names = {
    "SparqlCE": r"\textsc{SparqlCE}",
    "TF-IDF": "TF-IDF",
    "E5-base-v2": "E5-base-v2",
    "Qwen3-Embedding-0.6B": "Qwen3-Embedding-0.6B",
}


def latex_quality(value):
    """
    Format result quality for a LaTeX table.

    Undefined quality occurs when a configuration produces zero hits.
    """
    return (
        f"{value:.3f}"
        if value is not None
        else "--"
    )


print(
    f"% tab:caching — headline operating point "
    f"(K={K_h}, theta={theta_h}), "
    f"full grid in "
    f"results/caching/caching_dbpedia.json"
)


for name in order:

    r = all_results[name][
        f"K={K_h},theta={theta_h}"
    ]

    quality_str = latex_quality(
        r["result_quality"]
    )

    print(
        f"{latex_names[name]:<24} "
        f"& $({K_h}, {theta_h})$ "
        f"& {r['hit_rate']:.3f} "
        f"& {quality_str} \\\\"
    )


# ---------------------------------------------------------------------------
# Headline sentence draft
# ---------------------------------------------------------------------------
best = all_results[
    "SparqlCE"
][
    f"K={K_h},theta={theta_h}"
]


alts = {
    name: all_results[name][
        f"K={K_h},theta={theta_h}"
    ]
    for name in order[1:]
}


valid_alts = {
    name: result
    for name, result in alts.items()
    if result["result_quality"] is not None
}


print(
    "\n% headline sentence draft:"
)


if best["result_quality"] is None:

    print(
        f"% At K={K_h}, theta={theta_h}, "
        f"SparqlCE produces no cache hits, so "
        f"result quality is undefined."
    )


elif not valid_alts:

    print(
        f"% At K={K_h}, theta={theta_h}, "
        f"SparqlCE serves "
        f"{best['hit_rate']:.0%} of the stream "
        f"from cache with "
        f"{best['result_quality']:.2f} result quality. "
        f"No text baseline produces a cache hit at this "
        f"operating point."
    )


else:

    best_base = max(
        valid_alts,
        key=lambda name: (
            valid_alts[name][
                "result_quality"
            ]
        ),
    )

    print(
        f"% At K={K_h}, theta={theta_h}, "
        f"SparqlCE serves "
        f"{best['hit_rate']:.0%} of the stream "
        f"from cache with "
        f"{best['result_quality']:.2f} result quality, "
        f"vs. "
        f"{valid_alts[best_base]['result_quality']:.2f} "
        f"for the best text baseline "
        f"({best_base})."
    )

% tab:caching — headline operating point (K=100, theta=0.8), full grid in results/caching/caching_dbpedia.json
\textsc{SparqlCE}        & $(100, 0.8)$ & 0.710 & 0.273 \\
TF-IDF                   & $(100, 0.8)$ & 0.788 & 0.069 \\
E5-base-v2               & $(100, 0.8)$ & 0.997 & 0.017 \\
Qwen3-Embedding-0.6B     & $(100, 0.8)$ & 0.303 & 0.128 \\

% headline sentence draft:
% At K=100, theta=0.8, SparqlCE serves 71% of the stream from cache with 0.27 result quality, vs. 0.13 for the best text baseline (Qwen3-Embedding-0.6B).


### Notes

- The oracle (`result_sets_dbpedia.json`) replaces live endpoint execution, so
  the simulation is deterministic and offline.
- Rerunning with a different stream order: change `SEED` — report the default
  seed-42 numbers in the paper.
- Wikidata twin requires `datasets/result_sets_wikidata.json` (not in the
  repo yet).
